In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spikeinterface as si
from spikeinterface.preprocessing import notch_filter
from scipy.signal import coherence
import sys 

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT))

EPHYS_DIR = REPO_ROOT / "DATA" / "ephys"
DATA_DIR = REPO_ROOT / "DATA"

ANALYSIS_WINDOWS_FILE = DATA_DIR / "analysis_windows.csv"

NOTCH_FREQUENCY_HZ = 50
NOTCH_Q = 35

COHERENCE_WINDOW_S = 1.0
COHERENCE_OVERLAP = 0.5

OUTPUT_FILE = DATA_DIR / "lfp_coherence.csv"

In [ ]:
# Load the synchronized analysis windows used for coherence analysis

analysis_windows = pd.read_csv(
    ANALYSIS_WINDOWS_FILE
)

print(
    f"Loaded {len(analysis_windows):,} analysis windows."
)

recording_dirs = sorted(
    {
        path.parent.parent
        for path in EPHYS_DIR.rglob("lfp/meta.json")
    }
)

print(f"Found {len(recording_dirs)} recordings.")

In [ ]:
# Compute PFC-RSC coherence for each behavioral condition and recording

coherence_results = []

for recording_dir in recording_dirs:

    recording_name = recording_dir.name

    print(f"\nProcessing: {recording_name}")

    lfp_path = recording_dir / "lfp"

    recording = si.load(lfp_path)

    sampling_frequency = float(
        recording.get_sampling_frequency()
    )

    with (lfp_path / "meta.json").open("r") as file:
        metadata = json.load(file)

    pfc_channels = metadata["anatomical_assignment"]["pfc_channels"]
    rsc_channels = metadata["anatomical_assignment"]["rsc_channels"]

    traces = recording.get_traces()

    lfp_data = pd.DataFrame(
        traces,
        columns=recording.channel_ids,
    )

    regional_lfp = pd.DataFrame(
        {
            "PFC": lfp_data[pfc_channels].mean(axis=1),
            "RSC": lfp_data[rsc_channels].mean(axis=1),
        }
    )

    regional_recording = si.NumpyRecording(
        regional_lfp.values,
        sampling_frequency=sampling_frequency,
        channel_ids=["PFC", "RSC"],
    )

    filtered_recording = notch_filter(
        regional_recording,
        freq=NOTCH_FREQUENCY_HZ,
        q=NOTCH_Q,
    )

    recording_windows = analysis_windows[
        analysis_windows["recording"] == recording_name
    ]

    nperseg = int(
        round(
            sampling_frequency
            * COHERENCE_WINDOW_S
        )
    )

    noverlap = int(
        round(
            nperseg
            * COHERENCE_OVERLAP
        )
    )

    for label, condition_windows in recording_windows.groupby(
        "label"
    ):

        coherence_stack = []

        for _, window in condition_windows.iterrows():

            sample_start = int(window["sample_start"])
            sample_end = int(window["sample_end"])

            if sample_end <= sample_start:
                continue

            lfp_segment = filtered_recording.get_traces(
                start_frame=sample_start,
                end_frame=sample_end,
                channel_ids=["PFC", "RSC"],
            )

            if len(lfp_segment) < nperseg:
                continue

            frequencies, coherence_values = coherence(
                lfp_segment[:, 0],
                lfp_segment[:, 1],
                fs=sampling_frequency,
                nperseg=nperseg,
                noverlap=noverlap,
            )

            coherence_stack.append(
                coherence_values
            )

        if not coherence_stack:
            continue

        mean_coherence = np.mean(
            coherence_stack,
            axis=0,
        )

        for frequency, coherence_value in zip(
            frequencies,
            mean_coherence,
        ):

            coherence_results.append(
                {
                    "recording": recording_name,
                    "label": label,
                    "frequency_hz": frequency,
                    "coherence": coherence_value,
                }
            )

In [ ]:
# Assemble, save, and summarize the coherence results

coherence_df = pd.DataFrame(
    coherence_results
)

print(
    f"Generated {len(coherence_df):,} rows."
)

print(
    f"Recordings: "
    f"{coherence_df['recording'].nunique()}"
)

print(
    f"Conditions: "
    f"{sorted(coherence_df['label'].unique())}"
)

coherence_df.to_csv(
    OUTPUT_FILE,
    index=False,
)

print(f"Saved: {OUTPUT_FILE}")

coherence_summary = (
    coherence_df
    .groupby(
        ["label", "frequency_hz"],
        as_index=False,
    )
    .agg(
        mean_coherence=("coherence", "mean"),
        sem_coherence=(
            "coherence",
            lambda values: (
                values.std(ddof=1) / np.sqrt(len(values))
                if len(values) > 1
                else 0.0
            ),
        ),
        n=("coherence", "size"),
    )
)